# Day 5: Agentic AI – Building a Reasoning System

**Objective:** Understand agentic AI – how LLMs reason, use tools, and follow Model Context Protocols.

**Topics:**
- Agent design patterns (ReAct, Router)
- Building a simple LangChain ReAct agent with one tool (calculator)
- Creating a basic Model Context Protocol (MCP) server (get current time)
- Integrating your Day 3 vector database as an agent tool

> 💡 **Memory aid**: Agents = LLM + tools + loop. The LLM decides which tool to call, observes the result, and decides next step.

## 1. Setup & Imports

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, Tool, AgentType
from langchain.tools import tool
from langchain.utilities import PythonREPL
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate
from langchain.tools import BaseTool
from langchain.vectorstores import Pinecone as LangChainPinecone
from langchain.embeddings import OpenAIEmbeddings
import pinecone
import datetime
import json

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

print("Setup complete.")

## 2. Understanding Agent Patterns

Two key patterns:
- **ReAct (Reason + Act)**: Thought → Action → Observation loop.
- **Router**: Classify query and route to appropriate tool.

Today we focus on ReAct agents with LangChain.

## 3. Building a Simple ReAct Agent (Calculator Tool)

In [ ]:
# Define a calculator tool
@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression. Input should be a string like '2 + 2' or '10 * 3'."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

# Define a simple greeting tool
@tool
def greeting(name: str) -> str:
    """Returns a greeting message. Input should be a name."""
    return f"Hello, {name}! Welcome to the AI agent workshop."

# List of tools
tools = [calculator, greeting]
print(f"Created {len(tools)} tools: calculator, greeting")

In [ ]:
# Initialize LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Create a ReAct agent using the standard prompt template
from langchain.agents import create_react_agent
from langchain.prompts import PromptTemplate

react_prompt = PromptTemplate.from_template(
    """Answer the following question using the available tools. You must follow this format exactly:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}
"""
)

agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# Test
response = agent_executor.invoke({"input": "What is 25 multiplied by 4? Then greet me as Veera."})
print("\nFinal Answer:", response['output'])

**Observe:** The agent decides to call calculator first, gets the result, then calls greeting.

## 4. Building a Model Context Protocol (MCP) Server (Simple)

In [ ]:
# A simplified MCP-style server: a tool that provides context (current time)
@tool
def get_current_time(timezone: str = "UTC") -> str:
    """Returns the current date and time. Optionally specify timezone (default UTC)."""
    now = datetime.datetime.now(datetime.timezone.utc)
    if timezone.lower() == "ist":
        ist_offset = datetime.timedelta(hours=5, minutes=30)
        now = now + ist_offset
        return f"Current time in IST: {now.strftime('%Y-%m-%d %H:%M:%S')}"
    return f"Current time in UTC: {now.strftime('%Y-%m-%d %H:%M:%S')}"

# Another MCP-style tool: system status (mock)
@tool
def get_system_status() -> str:
    """Returns the system health status."""
    return json.dumps({"status": "healthy", "uptime": "99.99%", "active_connections": 42})

# Add to tools
tools.append(get_current_time)
tools.append(get_system_status)
print(f"Added MCP-style tools. Total tools: {len(tools)}")

# Recreate agent with new tools
agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# Test MCP tools
test_queries = [
    "What is the current time in IST?",
    "What is the system status?"
]
for q in test_queries:
    response = agent_executor.invoke({"input": q})
    print(f"\nQuery: {q}\nAnswer: {response['output']}\n" + "-"*50)

## 5. Integrating Your RAG Vector Database as an Agent Tool

This is powerful: the agent can choose to query the vector store when needed.

In [ ]:
# Connect to your Day 3 vector store
pc = pinecone.Pinecone(api_key=pinecone_api_key)
index_name = "rag-workshop"
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

vector_store = LangChainPinecone.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

@tool
def search_financial_docs(query: str) -> str:
    """Searches financial regulation documents (Basel III, GDPR, SOC2). Use this for questions about banking regulations, data protection, or compliance."""
    docs = vector_store.similarity_search(query, k=2)
    if not docs:
        return "No relevant documents found."
    results = []
    for i, doc in enumerate(docs):
        results.append(f"Document {i+1}: {doc.page_content[:300]}")
    return "\n\n".join(results)

# Add to tools list
tools.append(search_financial_docs)
print(f"Added RAG search tool. Total tools: {len(tools)}")

# Recreate agent with RAG tool
agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

In [ ]:
# Test: ask a regulation question that requires the vector DB
response = agent_executor.invoke({"input": "What are the key principles of GDPR?"})
print("\nFinal Answer:", response['output'])

# Test: mixed query (math + regulation)
response2 = agent_executor.invoke({"input": "What is 15% of 200? Also, what does Basel III say about capital buffers?"})
print("\nFinal Answer:", response2['output'])

## 6. Exercise: Create Your Own Tool

Create a tool that fetches live exchange rates (using a free API) and integrate it.

In [ ]:
# Example using free API (ExchangeRate-API or Frankfurter)
import requests

@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> str:
    """Get the current exchange rate between two currencies. Example: from_currency='USD', to_currency='EUR'"""
    try:
        url = f"https://api.frankfurter.app/latest?from={from_currency.upper()}&to={to_currency.upper()}"
        resp = requests.get(url)
        data = resp.json()
        rate = data['rates'][to_currency.upper()]
        return f"1 {from_currency.upper()} = {rate} {to_currency.upper()}"
    except Exception as e:
        return f"Error fetching exchange rate: {e}"

# Add to tools (uncomment to test)
# tools.append(get_exchange_rate)
# agent = create_react_agent(llm, tools, react_prompt)
# agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
# response = agent_executor.invoke({"input": "What is the exchange rate from USD to EUR?"})
# print(response['output'])

print("Exercise ready: uncomment the code above to add exchange rate tool.")

## 7. Understanding Agent Limitations & Best Practices

**Limitations:**
- Agents can loop infinitely (set `max_iterations`)
- Tools must have clear descriptions
- LLM may choose wrong tool

**Best practices:**
- Provide good tool descriptions
- Use `handle_parsing_errors=True`
- Set `max_iterations=5` to prevent loops
- Test with simple queries first

Example with safer configuration:

In [ ]:
safe_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=5,
    early_stopping_method="generate"
)
print("Safe agent executor created.")

## 8. Saving Your Agent Configuration

You can save the agent's tools and prompt for reuse.

In [ ]:
# Save tools to a file (optional)
import pickle
# with open('../src/agent_tools.pkl', 'wb') as f:
#     pickle.dump(tools, f)
print("Tools can be saved for later use.")

## Summary – Day 5 Completed ✅

You have learned:
- ReAct agent pattern and implementation in LangChain
- Building simple tools (calculator, greeting, time, status)
- Model Context Protocol (MCP) style tools for context provision
- Integrating your RAG vector database as an agent tool
- Agent limitations and best practices
- Creating a custom exchange rate tool (exercise)

**Next:** Day 6 – Production & Governance (security, real-time pipelines, compliance)

Save this notebook and commit to your repository.